# Limpieza COMPLETA de outputs (local + GCS)

⚠️ **Notebook destructivo, de uso manual.** Borra los outputs generados por el pipeline (local y en GCS) para poder rehacer una corrida desde cero. No forma parte del flujo reproducible de `notebooks/` — vive en `notebooks_legacy/` como utilidad de mantenimiento.

**No borra los audios originales** (`raw` / `raw_bajas` en GCS): solo prefijos de *outputs* de las fases 00–09 más las cachés locales de las demos (10/11).

Flujo:
1. Ejecutar la celda de configuración.
2. Ejecutar la auditoría (celda de solo lectura) y revisar qué se va a borrar.
3. Cambiar `CONFIRM_DELETE = True` en la celda de configuración únicamente cuando quieras borrar de verdad.
4. Ejecutar las celdas de borrado local y de GCS.


In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
# Las rutas y prefijos NO se hardcodean aquí: se importan de src/config.py,
# que a su vez los arma desde .env (TFM_GCS_BUCKET, TFM_GCS_PROJECT_PREFIX).
# Así la lista de carpetas/prefijos a limpiar queda sincronizada
# automáticamente con lo que cada fase realmente usa.

import sys
from pathlib import Path

# Busca la raíz del repo subiendo desde el directorio actual hasta encontrar
# la carpeta src/ -- no depende de dónde arranque el kernel de Jupyter
# (a diferencia de comparar el nombre de la carpeta actual).
CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR

while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "src").is_dir():
    raise RuntimeError(
        f"No se encontró la carpeta 'src' subiendo desde {CURRENT_DIR}. "
        "Verifica que Jupyter esté corriendo dentro del repositorio."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    DATA_DIR,
    EDA_DIR,
    CLEAN_RESULTS_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    TRANSCRIPTION_ROOT,
    PROXY_GROUNDTRUTH_DIR,
    SENTIMENT_DIR,
    PROSODY_DIR,
    SENTIMENT_FUSION_DIR,
    KEYWORD_DIR,
    VOICEPRINT_DIR,
    GCS_BUCKET,
    GCS_UNAV_CSV_PREFIX,
    GCS_UNAV_CLEAN_AUDIO_PREFIX,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    GCS_TRANSCRIPTION_PREFIX,
    GCS_PROXY_GROUNDTRUTH_PREFIX,
    GCS_SENTIMENT_PREFIX,
    GCS_PROSODY_PREFIX,
    GCS_SENTIMENT_FUSION_PREFIX,
    GCS_KEYWORD_PREFIX,
    GCS_VOICEPRINT_PREFIX,
    split_gcs_uri,
)

# Cachés locales de las demos (10/11) -- no suben nada a GCS, solo local.
from src.demo_end_to_end_audio_individual import DEMO_CACHE_DIR
from src.dashboard_resultados_globales import DASHBOARD_DIR

# ⚠️ Cambiar a True SOLO cuando ya se revisó la auditoría y se quiere borrar de verdad.
CONFIRM_DELETE = False

local_paths_to_clean = [
    EDA_DIR,
    CLEAN_RESULTS_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    TRANSCRIPTION_ROOT,
    PROXY_GROUNDTRUTH_DIR,
    SENTIMENT_DIR,
    PROSODY_DIR,
    SENTIMENT_FUSION_DIR,
    KEYWORD_DIR,
    VOICEPRINT_DIR,
    DEMO_CACHE_DIR,
    DASHBOARD_DIR,
]

# Prefijos de OUTPUTS en GCS. Deliberadamente NO incluye los audios de
# entrada (raw / raw_bajas): esos nunca se borran desde este notebook.
gcs_prefixes_to_clean = [
    GCS_UNAV_CSV_PREFIX,
    GCS_UNAV_CLEAN_AUDIO_PREFIX,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    GCS_TRANSCRIPTION_PREFIX,
    GCS_PROXY_GROUNDTRUTH_PREFIX,
    GCS_SENTIMENT_PREFIX,
    GCS_PROSODY_PREFIX,
    GCS_SENTIMENT_FUSION_PREFIX,
    GCS_KEYWORD_PREFIX,
    GCS_VOICEPRINT_PREFIX,
]

print("CONFIRM_DELETE:", CONFIRM_DELETE)
print(f"Carpetas locales a revisar: {len(local_paths_to_clean)}")
print(f"Prefijos GCS a revisar: {len(gcs_prefixes_to_clean)}")


In [ ]:
# ============================================================
# AUDITORÍA DE OUTPUTS ANTES DE BORRAR (solo lectura, no borra nada)
# ============================================================

from google.cloud import storage

print("=== LOCAL ===\n")

for local_path in local_paths_to_clean:
    if local_path.exists():
        n_files = sum(1 for _ in local_path.rglob("*") if _.is_file())
        print(f"EXISTE: {local_path}  | archivos: {n_files}")
    else:
        print(f"NO EXISTE: {local_path}")

print("\n=== GCS ===\n")

gcs_client = storage.Client()
bucket = gcs_client.bucket(GCS_BUCKET)

gcs_blob_counts = {}

for prefix_uri in gcs_prefixes_to_clean:
    _, prefix = split_gcs_uri(prefix_uri)
    blobs = list(bucket.list_blobs(prefix=prefix))
    gcs_blob_counts[prefix_uri] = len(blobs)
    print(f"{prefix_uri}: {len(blobs)} archivos")


In [ ]:
# ============================================================
# BORRADO REAL DE OUTPUTS LOCALES
# ============================================================

import shutil

if not CONFIRM_DELETE:
    print(
        "CONFIRM_DELETE=False -> no se borra nada. "
        "Cambia el flag en la celda de CONFIGURACIÓN para borrar de verdad."
    )
else:
    for local_path in local_paths_to_clean:
        if local_path.exists():
            shutil.rmtree(local_path)
            print("Borrado:", local_path)
        else:
            print("No existía:", local_path)

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    print("\nLimpieza local terminada.")


In [ ]:
# ============================================================
# BORRADO DE OUTPUTS EN GCS
# ============================================================
# Cada blob.delete() es una llamada de red -- con miles de archivos esto
# puede tardar minutos. Se usa tqdm para ver progreso real en vez de
# quedar en silencio hasta terminar.

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE=False -> no se borra nada en GCS.")
else:
    try:
        from tqdm.auto import tqdm
    except Exception:
        tqdm = None

    for prefix_uri in gcs_prefixes_to_clean:
        _, prefix = split_gcs_uri(prefix_uri)
        blobs = list(bucket.list_blobs(prefix=prefix))
        print(f"{prefix_uri}: borrando {len(blobs)} archivos...")

        iterator = (
            tqdm(blobs, desc=prefix_uri, unit="archivo")
            if tqdm is not None
            else blobs
        )

        for index, blob in enumerate(iterator, start=1):
            blob.delete()
            if tqdm is None and index % 200 == 0:
                print(f"  {index}/{len(blobs)} borrados...")

        print("Borrado:", prefix_uri)

    print("\nLimpieza GCS terminada.")
